# Clasificación Multi-Clase con BETO (BERT en español)
Pipeline completo: corpus → embeddings contextualizados [CLS] → Logistic Regression (OvR) → métricas por clase

## ¿Qué cambia respecto a TF-IDF?

| Elemento | TF-IDF (sesion_4) | BETO CLS (sesion_5) |
|---|---|---|
| Representación | Frecuencias de términos (sparse 1000-dim) | Embeddings contextualizados (dense **768-dim**) |
| Vocabulario | Limitado al corpus | Generaliza con BPE 31k subwords |
| `fit()` necesario | Sí — `fit_transform()` | **No** — modelo pre-entrenado |
| Multilingüe | No | **No** — especializado 100% en español |
| Contexto | Ninguno — `banco` siempre igual | **Sí** — `banco` varía según la oración |
| Estrategia del modelo | OvR | OvR — **sin cambios** |

> **⚠️ Nota importante:** `dccuchile/bert-base-spanish-wwm-uncased` es un modelo base de lenguaje (Fill-Mask / MLM),  
> **no tiene cabeza de clasificación**. No puede usarse directamente con `pipeline("text-classification")`.  
> En este notebook lo usamos como **extractor de features**: texto → token `[CLS]` (768-dim) → LogisticRegression.

In [ ]:
# Instalar librerías (solo necesario en Colab)
!pip install transformers torch -q

## 0 · Subir el corpus

In [ ]:
from google.colab import files
uploaded = files.upload()

## 1 · Imports y configuración

In [ ]:
import json, io
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

CLASES       = ['peliculas', 'restaurantes', 'productos', 'servicios', 'hoteles']
RANDOM_STATE = 42

## 2 · Carga del corpus

Sin cambios respecto a sesion_4: usamos `item['categoria']` como etiqueta.

In [ ]:
data = json.load(io.BytesIO(uploaded['corpus_sentimiento_reviews.json']))

reviews    = [item['texto']     for item in data['reviews']]
categorias = [item['categoria'] for item in data['reviews']]

print(f'Total de reseñas: {len(reviews)}')
print()
print('Ejemplos por categoría:')
for c in CLASES:
    print(f'  {c:<15}: {categorias.count(c):>3}')

## 3 · Métricas desde cero (sin sklearn)

Las **mismas funciones** de sesion_4, sin modificar.  
`pos_label` acepta cualquier string, así que funcionan con las 5 categorías.

In [ ]:
def confusion_matrix_manual(y_true, y_pred, pos_label='positivo'):
    TP = sum(t == pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    TN = sum(t != pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    FP = sum(t != pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    FN = sum(t == pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    return TP, TN, FP, FN

def accuracy(y_true, y_pred):
    TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
    return (TP + TN) / (TP + TN + FP + FN)

def precision(y_true, y_pred, pos_label='positivo'):
    TP, _, FP, _ = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall(y_true, y_pred, pos_label='positivo'):
    TP, _, _, FN = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1(y_true, y_pred, pos_label='positivo'):
    p = precision(y_true, y_pred, pos_label)
    r = recall(y_true, y_pred, pos_label)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

## 4 · Matriz de confusión N×N

Sin cambios. Para 5 clases la matriz es 5×5.

In [ ]:
def confusion_matrix_multiclase(y_true, y_pred, clases):
    """Imprime la matriz de confusión N×N en formato ASCII."""
    ancho = 14
    cabecera = ' ' * ancho + ''.join(c[:ancho].ljust(ancho) for c in clases)
    print(cabecera)
    print('-' * (ancho * (len(clases) + 1)))
    for real in clases:
        fila = real[:ancho].ljust(ancho)
        for pred in clases:
            count = sum(t == real and p == pred for t, p in zip(y_true, y_pred))
            fila += str(count).ljust(ancho)
        print(fila)

## 5 · Split Train / Test

Sin cambios. `stratify=categorias` garantiza la misma proporción en train y test.

In [ ]:
reviews_train, reviews_test, y_train, y_test = train_test_split(
    reviews, categorias, test_size=0.3, random_state=RANDOM_STATE,
    stratify=categorias
)

print(f'Train : {len(reviews_train)} reseñas')
print(f'Test  : {len(reviews_test)} reseñas')

## 6 · Cargar BETO

Se carga el **tokenizador** (convierte texto a IDs) y el **modelo base** (produce embeddings).  
`beto.eval()` desactiva el dropout para que los embeddings sean deterministas en inferencia.

In [ ]:
tokenizador = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')
beto        = AutoModel.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')
beto.eval()  # modo inferencia — desactiva dropout

print('Modelo cargado.')
print(f'Parámetros totales: {sum(p.numel() for p in beto.parameters()):,}')

## 7 · Función de codificación — token [CLS]

BERT agrega un token especial `[CLS]` al inicio de cada texto.  
Su vector en la última capa (`last_hidden_state[:, 0, :]`) condensa el significado global de la oración.  
Es el embedding que usamos como feature para el clasificador.

```
texto → [CLS] tok1 tok2 ... [SEP] → BETO → last_hidden_state[:, 0, :] → vector 768-dim
```

In [ ]:
def encode(textos, batch_size=32):
    """
    Pasa los textos por BETO y extrae el vector [CLS] (primer token).
    Retorna un array numpy de forma (n_textos, 768).
    """
    embeddings = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i : i + batch_size]
        tokens = tokenizador(
            lote,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        with torch.no_grad():                        # sin gradientes — solo inferencia
            salida = beto(**tokens)
        cls = salida.last_hidden_state[:, 0, :]     # primer token = [CLS]
        embeddings.append(cls.numpy())
    return np.vstack(embeddings)

## 8 · Embeddings y entrenamiento

Se llama `encode()` por separado en train y test — no existe `fit_transform()`  
porque BETO ya conoce el lenguaje. Se agrega `max_iter=500` para asegurar la convergencia  
con vectores de 768 dimensiones.

In [ ]:
print('Codificando train...')
X_train = encode(reviews_train)
print('Codificando test...')
X_test  = encode(reviews_test)

print(f'Dimensión de embeddings BETO: {X_train.shape[1]}')  # → 768

# El mismo clasificador OvR — solo cambia la representación de entrada
modelo = LogisticRegression(multi_class='ovr', random_state=RANDOM_STATE, max_iter=500)
modelo.fit(X_train, y_train)

print('Modelo entrenado.')
print(f'Clases aprendidas: {modelo.classes_.tolist()}')

## 9 · Evaluación con sklearn

`classification_report` muestra precision, recall y F1 **por clase** y el promedio macro.

In [ ]:
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred))

## 10 · Métricas manuales por clase

Las mismas funciones de sesion_4 aplicadas en un loop.

In [ ]:
print(f"{'Categoría':<15} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 48)

f1_por_clase = []
for c in CLASES:
    p = precision(y_test, y_pred, pos_label=c)
    r = recall(y_test, y_pred, pos_label=c)
    f = f1(y_test, y_pred, pos_label=c)
    f1_por_clase.append(f)
    print(f'{c:<15} {p:>10.2f} {r:>10.2f} {f:>10.2f}')

print('-' * 48)
macro_f1 = sum(f1_por_clase) / len(f1_por_clase)
print(f"{'Macro F1':<15} {'':>10} {'':>10} {macro_f1:>10.2f}")
print()
print(f'Accuracy global: {accuracy(y_test, y_pred):.2f}')

## 11 · Matriz de confusión 5×5

La diagonal principal son los aciertos. Fuera de la diagonal: errores por clase.

In [ ]:
confusion_matrix_multiclase(y_test, y_pred, CLASES)

## 12 · Predicción sobre textos nuevos

In [ ]:
nuevas_reviews = [
    'La actuación fue brillante y la historia muy emotiva',
    'El sushi estaba fresco y el servicio impecable',
    'El producto llegó en perfectas condiciones y funciona genial',
    'Tardaron semanas en responder y no solucionaron el problema',
    'Habitación limpia, cama cómoda y muy buena ubicación',
]

# encode() en lugar de vectorizador.transform()
preds = modelo.predict(encode(nuevas_reviews))

for texto, pred in zip(nuevas_reviews, preds):
    print(f'  [{pred.upper():<13}]  {texto}')